# Phase E — Global Context Modelling (`E18_Transformer` vs. `E19_Mamba`)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Controlled head-to-head comparison of **Self-Attention (`E18_Transformer`, `O(N^2 D)`)** vs. **2D Cross-Scan Selective SSM (`E19_Mamba`, `O(N D d_state)`)** on the exact same `16×16 = 256` spatial token grid inside `EnhancedPDN_S` (`200 epochs`, `AdamW cosine`, `patience=10`, native BF16).

**Prior Benchmarks:** `E01_PUAD_200ep` (`89.64%` / `84.07%`) | `E15_CrissCross` (`90.74%` / `85.33%`) | `E16_FPN` (`90.33%` / `87.84%`).

In [ ]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (PHASE E) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    if os.path.exists('/content/data/breakfast_box'):
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
    else:
        os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
        !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
else:
    os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'

if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

In [ ]:
#@title 3. Run Phase E (E18_Transformer & E19_Mamba) — Sequential for Clean Solo Latency
!python scripts/run_experiment.py --config configs/E18_transformer.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E19_mamba.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 4. Aggregate All Experiments (E01–E19 Master Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RUNS" --out-dir "$DRIVE_RUNS/phaseE_summary"